# Road Safety ML Greece — Exploratory Analysis, Benchmarking & Feature Ablation

**Project**: `road-safety-ml-greece` (v1.1.0 — **100% Official ELSTAT Data, Zero Synthetic Data**)  
**Primary Data Source**: Hellenic Statistical Authority (ELSTAT) — Survey `SDT04` (*Road Traffic Accidents resulting in Death or Injury*)

---

## Scientific Integrity & Scope
1. **100% Empirical Official Data**: Analyzes verified national annual totals (`2015–2024`) and official regional monthly panels (`13 NUTS-2 Regions × 60 Months = 780 observations`, `2021–2025`). Zero synthetic crash datasets or simulated Waze features are used.
2. **Strict Chronological Evaluation**: Evaluates `DummyClassifier`, `DecisionTree`, `RandomForest`, `HistGradientBoosting`, and `XGBoost` on a strictly chronological split (`2021–2023` Train, `2024` Val, `2025` Test), enforcing `validate_no_leakage(X)` and `assert_target_immutable(y_before, y_after)`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

nat_df = pd.read_csv(PROJECT_ROOT / 'data' / 'elstat_national_annual_2015_2024.csv')
print('Official National Totals (ELSTAT 2015-2024):')
print(nat_df[['year', 'total_injury_accidents', 'killed_30d', 'seriously_injured', 'slightly_injured', 'ksi_per_100_accidents']].to_string(index=False))

Official National Totals (ELSTAT 2015-2024):
 year  total_injury_accidents  killed_30d  seriously_injured  slightly_injured  ksi_per_100_accidents
 2015                   11440         793                999             13097                  15.66
 2016                   11318         824                879             12946                  15.05
 2017                   10848         731                706             12565                  13.25
 2018                   10737         700                727             12422                  13.29
 2019                   10712         688                652             12350                  12.51
 2020                    9083         584                518             10202                  12.13
 2021                   10454         624                610             11746                  11.80
 2022                   10487         654                664             11961                  12.57
 2023                   10553        

## 2. The Greek Road Safety Paradox: Urban Volume vs. Rural/Island Severity
In official ELSTAT statistics, **Attica** records more than half of all injury collisions in Greece (~53.5%), yet exhibits the lowest regional KSI severity rate (~6.5%) due to lower urban impact speeds and immediate tertiary trauma hospital access. Conversely, rural and island regions (**Crete, Peloponnese, South Aegean, Central Greece**) exhibit KSI rates exceeding 18%–24%.

In [2]:
reg_df = pd.read_csv(PROJECT_ROOT / 'data' / 'elstat_regional_monthly_panel_2021_2025.csv')
reg_df = reg_df[reg_df['year'].between(2021, 2025)]
summary_reg = (
    reg_df.groupby('region', as_index=False)
    .agg(total_accidents=('total_accidents', 'sum'), ksi_accidents=('ksi_accidents', 'sum'), mean_ksi_rate=('ksi_severity_rate', 'mean'))
    .sort_values('total_accidents', ascending=False)
)
summary_reg['mean_ksi_rate'] = summary_reg['mean_ksi_rate'].round(4)
print(summary_reg.head(8).to_string(index=False))

                                region  total_accidents  ksi_accidents  mean_ksi_rate
                       Attica (Αττική)            27965           1809         0.0652
Central Macedonia (Κεντρική Μακεδονία)             8625           1001         0.1166
        Western Greece (Δυτική Ελλάδα)             2732            525         0.1917
                           Crete (Κρήτη)           2512            592         0.2328
            Peloponnese (Πελοπόννησος)             2482            545         0.2191
           South Aegean (Νότιο Αιγαίο)             1931            478         0.2471
                   Thessaly (Θεσσαλία)             1803            348         0.1938
        Central Greece (Στερεά Ελλάδα)             1552            354         0.2272


## 3. Out-of-Time Model Benchmark & Feature Group Ablation (2025 Held-Out Test Set)
All models are trained strictly on `2021–2023`, calibrated on `2024`, and evaluated once on `2025` ($N_{\text{test}} = 156$ region-months).

In [3]:
bench_df = pd.read_csv(PROJECT_ROOT / 'reports' / 'benchmark_comparison.csv')
abl_df = pd.read_csv(PROJECT_ROOT / 'reports' / 'ablation_study.csv')
print('=== 5-Model Benchmark (2025 Held-Out Test Set) ===')
print(bench_df[['model', 'roc_auc', 'roc_auc_ci_95', 'pr_auc', 'f1', 'precision', 'recall', 'brier_score']].to_string(index=False))
print('\n=== Empirical Feature Group Ablation Study ===')
print(abl_df[['model', 'num_features', 'roc_auc', 'roc_auc_ci_95', 'pr_auc', 'f1', 'recall']].to_string(index=False))

=== 5-Model Benchmark (2025 Held-Out Test Set) ===
                     model  roc_auc    roc_auc_ci_95  pr_auc     f1  precision  recall  brier_score
DummyClassifier (Baseline)   0.5000 [0.5000, 0.5000]  0.4679 0.0000     0.0000  0.0000       0.2496
              DecisionTree   0.7958 [0.7209, 0.8642]  0.7522 0.7579     0.6154  0.9863       0.1903
              RandomForest   0.8567 [0.7973, 0.9105]  0.8374 0.7594     0.6228  0.9726       0.1552
      HistGradientBoosting   0.8199 [0.7509, 0.8808]  0.8036 0.7543     0.6471  0.9041       0.1919
                   XGBoost   0.8498 [0.7835, 0.9043]  0.8259 0.7609     0.6306  0.9589       0.1597

=== Empirical Feature Group Ablation Study ===
                                                             model  num_features  roc_auc    roc_auc_ci_95  pr_auc     f1  recall
                                    1. Calendar & Seasonality Only             4   0.5805 [0.4954, 0.6586]  0.5658 0.6376  1.0000
                                       2.